# Module 7: Running it for real, verification and field positioning

You run the regression set, one question for each failure you fixed in modules 1 to 6, against the assembled agent, and record the results in MLflow as a baseline. A later change to the model, the prompt or the tools can then be checked against it. You also take away a checklist for the field: when a customer says their agent misbehaves, it tells you which layer to look at first.

In [ ]:
# Load the lab's notebook helpers, and work in the agent's folder.
import labnb as lab
%cd ../agent

## Start from module 7

This cell puts the agent where module 7 starts. It copies any edits you made to
`agent/.reset-backup/` first, so nothing you wrote is lost.

The reset puts back every fix from modules 2 to 5. It also empties `gateway_url` in
[`config.toml`](../agent/config.toml), the agent's settings file, because module 6's change was an
edit you made by hand and the lab has no copy of it to restore. You don't need to edit `config.toml`
again. The cell after the reset writes the gateway address back in and prints the tools the agent
is offered. Your module 6 registration of `estimate_delivery` belongs to your project, so the reset
leaves it alone.

In [ ]:
# Reset the agent to module 7's starting state.
!python reset.py 7

In [ ]:
# Write the lab gateway's address into config.toml (the edit you made by hand in module 6),
# then print the tools the agent is offered.
import json
from pathlib import Path
gateway = lab.gateway_url()
if not gateway:
    raise RuntimeError("This workbench has no lab gateway address (the LAB_GATEWAY_URL environment "
                       "variable isn't set), so the agent can't reach its gateway tools. Ask for help before going on.")
config = Path("config.toml")
config.write_text(config.read_text().replace('gateway_url = ""', 'gateway_url = ' + json.dumps(gateway), 1))
lab.tool_list()

Look for `kestrel_estimate_delivery` in the list. The gateway puts `kestrel_` in front of the lab's
tool names, as you saw in module 6. The first four names, `discover_tools` to `select_tools`, have no
prefix because they're the gateway's own, not the lab's. The agent doesn't need them, and you can
ignore them.

If `kestrel_estimate_delivery` isn't there, add a cell below this one, run
`lab.register_tool("estimate_delivery")` in it, then run the cell above again. The tool server caches
registrations briefly, so a new one can take a short while to show up. Don't rerun the module 6
notebook to register it: its first cells reset the agent to module 6.

## 1. Run the regression set

The regression set is [`regression.py`](../agent/regression.py), a script that asks the agent one
question for each failure you fixed, then checks each run. It has six cases. Each is named after one
of the lab's questions in [`questions.py`](../agent/questions.py), the names you used with `--q`:

- `m1`, the K-400 coolant pump part number: the tool call fires (modules 1 and 2).
- `m3`, the coolant pump at the Denver depot: the loop uses the tool results and ends on the model's stop (module 3).
- `m4`, the sensor at the Portland depot: the failed stock check is reported (module 4).
- `m5`, fault F-21: the answer comes from a service bulletin and cites it (module 5).
- `m5-miss`, the K-900 service interval: no bulletin covers it, and the search says so (module 5).
- `m6`, delivery from Denver to Boulder: the agent uses the tool you registered at the gateway (module 6).

```mermaid
flowchart TB
  subgraph Cases["Six cases"]
    direction TB
    C1["m1: the part number"] ~~~ C3["m3: the Denver chain"] ~~~ C4["m4: the Portland depot"]
    C5["m5: fault F-21"] ~~~ C5M["m5-miss: the K-900"] ~~~ C6["m6: delivery time"]
  end
  Cases --> AG["The assembled agent"]
  AG --> REC["Run records: tools run, results, how each run ended"]
  REC --> J{"Fixed checks in regression.py"}
  J --> REP["Report, and an evaluation run in MLflow"]
```

Each case is checked by fixed rules in `regression.py`, against the run's record: which tools ran,
what they returned and how the run ended. No second model grades the answer, so the same run always
gets the same verdict. Two cases also look for words in the answer: `m5` must cite `SB-2024-102`, and
`m4` must say the stock couldn't be checked. A word check like that can miss a correct answer that's
worded another way.

Here's the `m5` check:

In [ ]:
# Show the m5 case's check, check_m5, from regression.py.
lab.show_code("regression.py", "check_m5")

It reports SKIP if the agent isn't offered `search_bulletins`, FAIL if no passage from SB-2024-102
came back or the answer doesn't name it, and PASS otherwise.

The next cell runs all six cases and records them in MLflow under the name `my-baseline`. It takes a
few minutes, and nothing prints until the last case has finished. Wait for it rather than running it
again.

In [ ]:
# Run all six cases against the agent, and record them in MLflow as an evaluation run named
# my-baseline. Nothing prints until the last case finishes.
lab.regression(name="my-baseline")

Each case prints a verdict and the reason for it:

- `PASS`: the run did what that module's fix makes it do.
- `FAIL`: it didn't, and the reason says what the check didn't find. A case whose run was cut off
  fails too. If the model runs out of room before it finishes its answer, the run ends on `length`,
  and the case is FAIL whatever else its check found. The reason says to run the set again.
- `SKIP`: the case can't apply, because the agent isn't offered the tool it needs. On an earlier
  module's agent, `m5` and `m5-miss` would skip without module 5's `search_bulletins`, and `m6`
  without the delivery tool from module 6. Here every fix is in place, so a SKIP means something is
  missing. See *If the result differs*.

**This is the module's success signal:** all six regression cases PASS, with 0 FAIL and 0 SKIP.

## 2. Check the evidence, not the answers

A PASS says the check found what it looked for. The run records show what the agent actually did.
The table below has one row per case, in the order they ran, and the question column tells you which
is which. Click a run's name to open its trace in MLflow, where you can read the whole answer.

What each row should show:

- `m1`, the part number: `kestrel_find_part (ok)`.
- `m3`, Denver: `kestrel_find_part (ok)`, then `kestrel_check_stock (ok)`. The model may check the seal kit's stock too.
- `m4`, Portland: `kestrel_check_stock (bad_request)` after the part lookup, and the answer says Portland isn't a valid depot.
- `m5`, F-21: `kestrel_search_bulletins (ok)`, and the answer cites SB-2024-102.
- `m5-miss`, the K-900: `kestrel_search_bulletins (no_results)`, and the answer says the bulletins don't cover it.
- `m6`, Boulder: `kestrel_estimate_delivery (ok)`.

Every row should end on `stop`, the model finishing on its own. A row that ends on `length` is a run
the model cut off, and its case is FAIL (see step 1). The table shows the six newest runs, so if you
ran the set twice, it shows the second run.

In [ ]:
# Show a table of the six newest runs (one per case), one row each:
#   question: the question asked
#   turns: how many times the agent called the model
#   ended by: what stopped the run
#   tools run: the tools the agent ran, and each result's category
#   answer: the start of the agent's answer
lab.evidence(columns=("question", "turns", "ended by", "tools run", "answer"), last=6)
# A link to the list of all your traces in MLflow.
lab.links(("Every trace in this experiment", lab.experiment_url()))

## 3. The baseline in MLflow

`name="my-baseline"` in step 1 also recorded the run in MLflow as an **evaluation run**: one row per
case, each with its trace and the same verdict and reason the cell printed. MLflow keeps each
verdict on the case's trace as an assessment named `check`. The run also keeps the counts, `passed`,
`failed` and `skipped`, and it's linked to the version of the agent that ran it. Yours is
`kestrel-m7-start`, the name for module 7's starting state. If you'd edited the agent since the
reset, it would read `kestrel-m7-start-modified`.

Open the link below. It lists your evaluation runs, with a row for `my-baseline`. Select it to see its
six cases, each with its verdict, reason and trace. The counts on the experiment's Runs tab should
match the regression cell's last line: 6 passed, 0 failed, 0 skipped.

In [ ]:
# A link to your evaluation runs in MLflow.
lab.links(("Your evaluation runs in MLflow", lab.experiment_url("evaluation-runs")))

## 4. Compare a second run with the baseline (stretch goal)

Skip this step if you're short of time.

A baseline is only useful against a later run. After a change, such as a new model version or a
prompt edit, you run the set again under a new name and compare the two. To try the comparison now,
add a cell below this one and run `lab.regression(name="second-run")` in it. It takes as long as the
first run. Then open the evaluation runs link above and select both `my-baseline` and `second-run`.
MLflow shows "Comparing" the two runs, with each case's two traces and verdicts side by side and a
summary of which verdicts changed.

You changed nothing in between, so expect the same six PASS results. A case that differs is one the
model handled differently this time.

## 5. Symptom to layer

The regression run also wrote a report under `agent/runs/`. It repeats the six results, then ends
with the checklist this lab gives you to take away: what a customer might say about their agent, and
the layer to look at first. The checklist is the part to read.

In [ ]:
# Show the newest regression report from agent/runs/.
reports = sorted(lab.RUNS.glob("regression-*.md"))
lab.display(lab.Markdown(reports[-1].read_text() if reports else "No report yet."))

Each row maps to a module you worked through. The last row, "Who can see what?", is module 6. When
you registered `estimate_delivery`, only your own project's tool catalog changed, not other
participants'. The platform decided which tools each agent was offered, not the agent's code.

**Your read:** a customer says their agent told them it ordered a part, but no order was ever placed.
Which layer do you look at first?

<details><summary>Check your answer</summary>

Tool returns (module 4). It's the "confidently reports work it did not do" row: check whether the
ordering tool's result can say that it failed. In module 4, an unknown depot came back looking like
"no stock" until the tool returned `bad_request`. Here's the whole checklist:

| A customer says... | Look first at | Layer |
|---|---|---|
| The agent ignores its tools | How the model is served (tool-call parser), not the prompt | Model interface (module 2) |
| It loops, or runs are expensive | Whether the loop appends tool results and reads the stop condition | Agent loop (module 3) |
| It confidently reports work it did not do | Whether tool returns can express failure | Tool returns (module 4) |
| It makes things up about their own data | Whether anything grounds it in a source | Knowledge search (module 5) |
| Who can see what? | Whether tool access goes through a governed path | Platform tool access (module 6) |
</details>

The report is saved in `agent/runs/` as `regression-<time>.md`. To keep it, right-click it in the
file browser and choose **Download**. The checklist is also on this module's page in Showroom, in the
answer above.

This baseline is evidence that today's build handles the failures it was built for, and no evidence
at all about failures nobody has thought of yet.

Module 5 said nothing yet checked that the retrieved passage answers the question. This set doesn't
either. The `m5` check confirms that the answer cites SB-2024-102, not that what it says is right.
Checking that takes a person, or a model grading the answer, and this lab leaves it out.

The report's last lines name two more failures this lab doesn't test. The first is the model or the
prompt changing underneath an agent that works. Rerunning this set after a change like that is how
you'd catch it, and that's what the baseline is for. The second is cost when many people use the
agent at once. These checks ask one question at a time, so they can't show it.

## If the result differs

Success is 6 PASS, 0 FAIL, 0 SKIP. A cell that finishes without an error doesn't mean its cases
passed, so read each case's verdict and reason.

If `m6` shows SKIP, the agent wasn't offered the delivery tool. Run the gateway cell under *Start from
module 7* again and look for `kestrel_estimate_delivery` in its list. If it's missing, register it as
that section describes, then run the regression set again.

If a case fails because its run was cut off, the model used up its allowance of 4,096 tokens for one
model call before it finished its answer. The allowance counts the model's reasoning as well as its
answer. Run the set again.

If `m1` or `m3` fails, check which model endpoint the agent calls: add a cell and run
`print(lab.endpoint())`. It should start with `qwen3-8b-b`, the endpoint module 2 moved the agent to.
If it doesn't, run the reset cell and the gateway cell again.

For any other FAIL, the reason says what the check didn't find. Open that case's trace from the table
in step 2 to see what the agent did instead. For `m5`, the reason tells you whether SB-2024-102 wasn't
retrieved at all, or was retrieved but left out of the answer.

If the regression cell's link says the evaluation run isn't available, read the run's last lines. If
they say the evaluation wasn't recorded in MLflow, every result is still in the report. Run the set
again once MLflow is reachable, before you compare runs.

If a rerun gives different results, check the endpoint as above, and check that `gateway_url` in
`config.toml` still holds the gateway address. `config.toml` keeps its contents when the workbench
restarts. The reset cell puts it back to module 7's start, and the gateway cell after it sets the
address again.